In [1]:
import pandas as pd
import yaml

# ---------------------------------
# Display settings
# ---------------------------------
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.float_format', '{:,.0f}'.format)

# ------------------------------
# Load settings and initialize
# ------------------------------
with open('../../Settings.yaml', 'r') as file:
    Setting = yaml.safe_load(file)

Data = []

# ------------------------------
# Load Temperature Dataset
# ------------------------------
for year in range(Setting['Start_Year'], Setting['End_Year'] + 1 ):
    file_name = f'Y{year}_Temprature.xlsx'
    file_path = f"{Setting['Raw_Path']}/{file_name}"
    Temp = pd.read_excel(file_path, skiprows=1)
    Temp.columns = ['Province Center','TempAvgMax','TempAvgMin','TempAbsMax',
                    'TempAbsMin','TempAvg','AnnualRain','MaxDailyRain',
                    'HumAvgMax','HumAvgMin','FrostDays','SunHours','MaxWindSpeed_ms']
    Temp['Year'] = year
    Data.append(Temp)

Temprature = pd.concat(Data, ignore_index=True)
Temprature['Province Center'] = (
    Temprature['Province Center'].astype(str)
    .str.strip() 
    .str.replace("ي", "ی", regex=False)
    .str.replace("ك", "ک", regex=False)
    .str.replace('بندر عباس', 'بندرعباس', regex=False)
    .str.replace('خرم\u200cآباد', 'خرم آباد', regex=False)
)

# ------------------------------
# Load Province info Dataset
# ------------------------------
file_name_01 = 'Province_Info.xlsx'
file_path_01 = f"{Setting['Raw_Path']}/{file_name_01}"
Province_Info = pd.read_excel(file_path_01)
Province_Info.columns = ['Province','Province Center','Area',
    'County No','District No','City No','RuralDistrict No']

Province_Info['Province Center'] = (
    Province_Info['Province Center'].astype(str)
    .str.strip() 
    .str.replace("ي", "ی", regex=False)
    .str.replace("ك", "ک", regex=False)
    )

Dataset = pd.merge(
    Temprature,
    Province_Info,
    how='outer',
    on = 'Province Center'
)
Dataset.drop(columns=['TempAvgMax', 'TempAvgMin', 'TempAbsMax',
                      'TempAbsMin','MaxDailyRain', 'HumAvgMax',
                      'HumAvgMin', 'FrostDays','MaxWindSpeed_ms',
                      'County No', 'District No', 'City No',
                      'RuralDistrict No'],inplace=True) 

# ------------------------------------------------------------
# Replicate Tehran climate values for Alborz (1381–1389)
# but set Area from Karaj/Alborz (not Tehran)
# ------------------------------------------------------------

target_years = range(1381, 1390)
tehran_mask = (Dataset['Province Center'] == 'تهران') & (Dataset['Year'].isin(target_years))
tehran_records = Dataset.loc[tehran_mask].copy()
alborz_records = tehran_records.copy()
alborz_records['Province'] = 'البرز'
alborz_records['Province Center'] = 'کرج'
karaj_area = Dataset.loc[Dataset['Province Center'] == 'کرج', 'Area'].dropna()
alborz_records['Area'] = karaj_area.iloc[0]
Dataset = pd.concat([Dataset, alborz_records], ignore_index=True)

# ------------------------------------------------------------
# Replicate  Khorasan Razavi climate values for  North Khorasan  (1381–1382)
# ------------------------------------------------------------

target_years = range(1381, 1383)
mashhad_mask = (Dataset['Province Center'] == 'مشهد') & (Dataset['Year'].isin(target_years))
mashhad_records = Dataset.loc[mashhad_mask].copy()
bojnord_records = mashhad_records.copy()
bojnord_records['Province'] = 'خراسان شمالی'
bojnord_records['Province Center'] = 'بجنورد'
bojnord_area = Dataset.loc[Dataset['Province Center'] == 'بجنورد', 'Area'].dropna()
bojnord_records['Area'] = bojnord_area.iloc[0]
Dataset = pd.concat([Dataset, bojnord_records], ignore_index=True)

# ------------------------------------------------------------
# Replicate  Khorasan Razavi climate values for  South Khorasan  (1381–1382)
# ------------------------------------------------------------

target_years = range(1381, 1383)
mashhad_mask = (Dataset['Province Center'] == 'مشهد') & (Dataset['Year'].isin(target_years))
mashhad_records = Dataset.loc[mashhad_mask].copy()
birjand_records = mashhad_records.copy()
birjand_records['Province'] = 'خراسان جنوبی'
birjand_records['Province Center'] = 'بیرجند'
birjand_area = Dataset.loc[Dataset['Province Center'] == 'بیرجند', 'Area'].dropna()
birjand_records['Area'] = birjand_area.iloc[0]
Dataset = pd.concat([Dataset, birjand_records], ignore_index=True)

# ------------------------------
# Final check
# ------------------------------

Dataset['SunHours'] = (Dataset['SunHours'].replace('-', 0)) 
Dataset = Dataset[['Year', 'Province', 'Province Center', 'Area', 'TempAvg', 'AnnualRain', 'SunHours']]
Dataset.sort_values(by=['Year','Province'],inplace=True)

# ------------------------------
# Export Province Information 
# ------------------------------
output_file_name = 'General.xlsx'
sheet_name= 'Province_Info'
path = f"{Setting['Output_Path_General']}/{output_file_name}"
with pd.ExcelWriter(path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    Province_Info.to_excel(writer,sheet_name=sheet_name ,index=False)

# ------------------------------
# Export Temprature by province 
# ------------------------------
output_file_name = 'General.xlsx'
sheet_name= 'Weather_Province_Year'
path = f"{Setting['Output_Path_General']}/{output_file_name}"
with pd.ExcelWriter(path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    Dataset.to_excel(writer,sheet_name=sheet_name ,index=False)


C:\Users\hwi\AppData\Local\Temp\ipykernel_7832\3082314983.py:117: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  Dataset['SunHours'] = (Dataset['SunHours'].replace('-', 0))
